# Synthetic Benchmark

Runs all four approaches on Ω₁ = {1, 1.2, 3} and Ω₂ = {11, 11.2, 13},
plus a frequency offset robustness sweep.

Produces:
- `fig:unary_lines` — qualitative function fitting comparison
- `fig:r2s_jadGaussianShift` — R² vs frequency offset
- `tab:baseline_comparison` data — combined with CMA-ES from notebook 03

In [1]:
# ── Smoke test flag ────────────────────────────────────────────
SMOKE_TEST = True

from pathlib import Path
import numpy as np
import jax
import jax.numpy as jnp
import optax
import pennylane as qml
import pandas as pd
import pickle
import sys
from sklearn.metrics import r2_score

sys.path.append("..")
from paper_style import apply_paper_style, BLUE, RED, GREEN, ORANGE, PURPLE, APPROACH_COLORS
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

jax.config.update("jax_enable_x64", True)
apply_paper_style()
SEED = 42
np.random.seed(SEED)

## Hyperparameters

In [2]:
NUM_FUNCTIONS  = 1 if SMOKE_TEST else 10
NUM_RUNS       = 2 if SMOKE_TEST else 10
MAX_STEPS      = 50  if SMOKE_TEST else 5000
LR             = 0.001
BATCH_SIZE     = 40
NUM_WIRES      = 3
NUM_SERIAL_LAYERS  = 2
TRAINABLE_BLOCKS   = 1
NUM_SU_GATES       = 1
NUM_ROT_PARAMS     = 63  # SU(8)

DATA_DIR = Path("../datasets")
OUT_DIR  = Path("results")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Approach configurations: (label, initial_alpha, fixed_alpha)
# fixed_alpha=True means alpha is NOT trained (Fixed Unary / Fixed Ternary)
APPROACHES = [
    ("Fixed Unary",      [[1.0, 1.0, 1.0]],   True),
    ("Trainable Unary",  [[1.01, 1.02, 1.03]], False),
    ("Fixed Ternary",    [[1.0, 3.0, 9.0]],    True),
    ("Trainable Ternary",[[1.0, 3.0, 9.0]],    False),
]

print(f"SMOKE_TEST={SMOKE_TEST}  functions={NUM_FUNCTIONS}  "
      f"runs={NUM_RUNS}  steps={MAX_STEPS}")

SMOKE_TEST=True  functions=1  runs=2  steps=50


## Circuit and training helpers

In [3]:
def S(alpha, x, num_wires, enc_layer):
    for w in range(num_wires):
        qml.RX(alpha[w] * x, wires=w)

def W_SU(theta, trainable_block_layers, num_wires):
    for i in range(trainable_block_layers):
        qml.SpecialUnitary(theta[i][0], wires=range(num_wires))

def weights_ones(n_serial, t_blocks, n_su, n_params):
    return np.ones((n_serial, t_blocks, n_su, n_params))

def square_loss(targets, predictions):
    return 0.5 * sum((t-p)**2 for t,p in zip(targets, predictions)) / len(targets)

dev = qml.device("default.qubit", wires=NUM_WIRES)

@qml.qnode(dev, interface="jax")
def circuit(alpha, weights_SU, x=None):
    W_SU(weights_SU[0], TRAINABLE_BLOCKS, NUM_WIRES)
    for i in range(NUM_SERIAL_LAYERS - 1):
        S(alpha[i], x, NUM_WIRES, i)
        W_SU(weights_SU[i+1], TRAINABLE_BLOCKS, NUM_WIRES)
    return qml.expval(qml.PauliZ(wires=NUM_WIRES-1))

circuit_jit = jax.jit(circuit)


def run_approach(label, alpha_init, fixed_alpha,
                 datasets, num_functions, num_runs,
                 max_steps, lr, batch_size):
    """Train one approach configuration and return a results DataFrame."""
    results = []
    for func in range(num_functions):
        ds = datasets[func]
        x_tr, y_tr = ds["x_train"], ds["y_train"]
        x_te, y_te = ds["x_test"],  ds["y_test"]

        for run in range(num_runs):
            alpha   = jnp.array(alpha_init)
            weights = jnp.array(weights_ones(
                NUM_SERIAL_LAYERS, TRAINABLE_BLOCKS,
                NUM_SU_GATES, NUM_ROT_PARAMS
            ))
            if fixed_alpha:
                # Only train ansatz weights
                params    = {"weights_SU": weights}
                opt       = optax.adam(lr)
                opt_state = opt.init(params)

                def cost_fn(p, x, y):
                    preds = [circuit_jit(alpha, p["weights_SU"], x_)
                             for x_ in x]
                    return square_loss(y, preds).squeeze()
            else:
                # Train both ansatz weights and prefactors
                params    = {"weights_SU": weights, "alpha": alpha}
                opt       = optax.adam(lr)
                opt_state = opt.init(params)

                def cost_fn(p, x, y):
                    preds = [circuit_jit(p["alpha"], p["weights_SU"], x_)
                             for x_ in x]
                    return square_loss(y, preds).squeeze()

            @jax.jit
            def update(p, s, x, y):
                loss, g = jax.value_and_grad(cost_fn)(p, x, y)
                u, s2   = opt.update(g, s, p)
                return optax.apply_updates(p, u), s2, loss

            for step in range(max_steps):
                idx = np.random.randint(0, len(x_tr), batch_size)
                params, opt_state, c = update(
                    params, opt_state,
                    jnp.array(x_tr[idx]), jnp.array(y_tr[idx])
                )

            final_alpha = params.get("alpha", alpha)
            w_final     = params["weights_SU"]
            preds = [float(circuit_jit(final_alpha, w_final, x_))
                     for x_ in x_te]
            r2 = r2_score(y_te, preds)
            print(f"  {label}  func={func+1}  run={run+1}  R²={r2:.4f}")
            results.append({
                "Approach": label, "Func": func, "Run": run,
                "R2_Score": r2, "Cost": float(c),
                "final_alpha": final_alpha.tolist()
            })
    return pd.DataFrame(results)

## Experiment 1: Ω₁ (Jaderberg original)

In [ ]:
with open(DATA_DIR / "datasets_1d_jaderberg.pkl", "rb") as f:
    ds_omega1 = pickle.load(f)

# Squeeze all arrays to remove the trailing (1,) dimension
for ds in ds_omega1:
    ds["x_train"] = np.array(ds["x_train"]).squeeze()
    ds["x_test"]  = np.array(ds["x_test"]).squeeze()
    ds["y_train"] = np.array(ds["y_train"]).squeeze()
    ds["y_test"]  = np.array(ds["y_test"]).squeeze()

dfs_omega1 = []
for label, alpha_init, fixed in APPROACHES:
    print(f"\n── Ω₁  {label} ──")
    df = run_approach(label, alpha_init, fixed,
                      ds_omega1, NUM_FUNCTIONS, NUM_RUNS,
                      MAX_STEPS, LR, BATCH_SIZE)
    dfs_omega1.append(df)

omega1_df = pd.concat(dfs_omega1, ignore_index=True)
omega1_df["target"] = "Omega1"
omega1_df.to_csv(OUT_DIR / "synthetic_omega1.csv", index=False)


── Ω₁  Fixed Unary ──
  Fixed Unary  func=1  run=1  R²=0.9004
  Fixed Unary  func=1  run=2  R²=0.9055

── Ω₁  Trainable Unary ──
  Trainable Unary  func=1  run=1  R²=0.9021
  Trainable Unary  func=1  run=2  R²=0.9005

── Ω₁  Fixed Ternary ──


## Experiment 2: Ω₂ (shifted +10)

In [ ]:
with open(DATA_DIR / "datasets_1d_jaderberg_10.pkl", "rb") as f:
    ds_omega2 = pickle.load(f)

# Squeeze all arrays to remove the trailing (1,) dimension
for ds in ds_omega2:
    ds["x_train"] = np.array(ds["x_train"]).squeeze()
    ds["x_test"]  = np.array(ds["x_test"]).squeeze()
    ds["y_train"] = np.array(ds["y_train"]).squeeze()
    ds["y_test"]  = np.array(ds["y_test"]).squeeze()

dfs_omega2 = []
for label, alpha_init, fixed in APPROACHES:
    print(f"\n── Ω₂  {label} ──")
    df = run_approach(label, alpha_init, fixed,
                      ds_omega2, NUM_FUNCTIONS, NUM_RUNS,
                      MAX_STEPS, LR, BATCH_SIZE)
    dfs_omega2.append(df)

omega2_df = pd.concat(dfs_omega2, ignore_index=True)
omega2_df["target"] = "Omega2"
omega2_df.to_csv(OUT_DIR / "synthetic_omega2.csv", index=False)

## Experiment 3: Frequency offset robustness sweep

In [ ]:
with open(DATA_DIR / "datasets_1d_jaderberg_GaussianShift.pkl", "rb") as f:
    ds_shift = pickle.load(f)   # dict: {mu_float: list_of_datasets}

# Squeeze all arrays to remove the trailing (1,) dimension
for ds in ds_shift:
    ds["x_train"] = np.array(ds["x_train"]).squeeze()
    ds["x_test"]  = np.array(ds["x_test"]).squeeze()
    ds["y_train"] = np.array(ds["y_train"]).squeeze()
    ds["y_test"]  = np.array(ds["y_test"]).squeeze()

shift_results = []
mu_values = sorted(ds_shift.keys())

for mu in mu_values:
    ds = ds_shift[mu]
    for label, alpha_init, fixed in APPROACHES:
        print(f"offset μ={mu:.1f}  {label}")
        df = run_approach(label, alpha_init, fixed,
                          ds, NUM_FUNCTIONS, NUM_RUNS,
                          MAX_STEPS, LR, BATCH_SIZE)
        df["mu"] = mu
        shift_results.append(df)

shift_df = pd.concat(shift_results, ignore_index=True)
shift_df.to_csv(OUT_DIR / "synthetic_offset_sweep.csv", index=False)
print("Offset sweep done.")

## Plots

In [ ]:
from paper_style import boxplot_panel

omega2_df = pd.read_csv(OUT_DIR / "synthetic_omega2.csv")

approach_order  = ["Fixed Unary", "Trainable Unary",
                   "Fixed Ternary", "Trainable Ternary"]
approach_colors = [RED, BLUE, ORANGE, GREEN]
data = {a: omega2_df[omega2_df.Approach==a].R2_Score.values
        for a in approach_order}

fig, ax = plt.subplots(figsize=(3.5, 3.0))
boxplot_panel(ax, data, approach_colors)
plt.tight_layout()
plt.savefig("r2_omega2_boxplot.pdf", dpi=600, bbox_inches="tight")
plt.show()
print("Saved r2_omega2_boxplot.pdf")

In [ ]:
# fig:r2s_jadGaussianShift — median R² vs frequency offset
shift_df = pd.read_csv(OUT_DIR / "synthetic_offset_sweep.csv")

fig, ax = plt.subplots(figsize=(3.3, 2.8))

for label, color in zip(approach_order, approach_colors):
    sub = shift_df[shift_df.Approach == label]
    stats = sub.groupby("mu")["R2_Score"].agg(
        median="median",
        q25=lambda x: np.percentile(x, 25),
        q75=lambda x: np.percentile(x, 75)
    ).reset_index()
    ax.plot(stats["mu"], stats["median"],
            label=label, color=color, linewidth=1.5)
    ax.fill_between(stats["mu"], stats["q25"], stats["q75"],
                    alpha=0.20, color=color)

ax.axhline(0.95, color="black", linestyle="--", linewidth=0.9,
           label=r"$R^2\!=\!0.95$")
ax.set_xlabel(r"Frequency offset $\mu$", fontsize=9, fontweight="bold")
ax.set_ylabel(r"Median $R^2$ (test set)", fontsize=9, fontweight="bold")
ax.legend(loc="lower left", frameon=True, fancybox=False,
          shadow=False, fontsize=7)
ax.grid(True, alpha=0.5, linestyle="-", linewidth=0.8)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig("r2_vs_freqshift.pdf", dpi=600, bbox_inches="tight")
plt.savefig("r2_vs_freqshift.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved r2_vs_freqshift.pdf")

## Summary statistics

In [ ]:
print("Ω₂ results:")
print(f"{'Approach':<20} {'N':>4} {'Median':>7} {'Success%':>9}")
print("-"*44)
for a in approach_order:
    v = data[a]
    print(f"{a:<20} {len(v):>4} {np.median(v):>7.3f} "
          f"{np.mean(v>=0.95)*100:>8.0f}%")